# 01 — `*args`, `**kwargs`, and the full signature grammar

Python function signatures can express a lot more than `def f(a, b)`. Mastering this is what lets you read FastAPI / Pydantic / agent-framework code without squinting.

## Positional vs keyword arguments — recap

Most parameters can be passed either way. The caller chooses.

In [1]:
def greet(name, greeting="Hello"):
    return f"{greeting}, {name}!"

print(greet("Dinakar"))                       # positional
print(greet("Dinakar", "Hi"))                  # positional
print(greet(name="Dinakar", greeting="Hey"))   # keyword
print(greet(greeting="Yo", name="Dinakar"))    # keyword, any order

Hello, Dinakar!
Hi, Dinakar!
Hey, Dinakar!
Yo, Dinakar!


## `*args` — collect extra positional args into a tuple

In [2]:
def total(*nums):
    print("received:", nums, type(nums))
    return sum(nums)

print(total(1, 2, 3))
print(total())          # zero args is fine — empty tuple
print(total(10, 20))

received: (1, 2, 3) <class 'tuple'>
6
received: () <class 'tuple'>
0
received: (10, 20) <class 'tuple'>
30


## `**kwargs` — collect extra keyword args into a dict

In [3]:
def configure(**options):
    print("received:", options, type(options))
    for k, v in options.items():
        print(f"  {k} = {v}")

configure(debug=True, timeout=30, theme="dark")

received: {'debug': True, 'timeout': 30, 'theme': 'dark'} <class 'dict'>
  debug = True
  timeout = 30
  theme = dark


## The full grammar — combined

Order is fixed:

```python
def f(positional, default=1, *args, kw_only, kw_default=2, **kwargs):
    ...
```

After `*args` (or a bare `*`), **everything is keyword-only.**

In [4]:
def log(message, *args, level="INFO", **fields):
    extras = " ".join(f"{k}={v}" for k, v in fields.items())
    print(f"[{level}] {message} {args} {extras}")

log("login failed")
log("login failed", "alice", "192.168.1.1")
log("login failed", level="ERROR", user="alice", attempts=3)
log("login failed", "alice", level="WARN", reason="bad pw")

[INFO] login failed () 
[INFO] login failed ('alice', '192.168.1.1') 
[ERROR] login failed () user=alice attempts=3
[WARN] login failed ('alice',) reason=bad pw


## Forcing keyword-only — the bare `*`

Sometimes you want params *after* the positionals to be **keyword-only**, but you don't want to accept `*args`. Use a bare `*` as a delimiter.

In [5]:
def make_user(name, *, is_admin=False, email=None):
    # is_admin and email MUST be passed by keyword. This prevents
    # the bug:  make_user("Dinakar", True)  -- is True the admin flag or the email?
    return {"name": name, "is_admin": is_admin, "email": email}

print(make_user("Dinakar", is_admin=True))

try:
    make_user("Dinakar", True)     # rejected
except TypeError as e:
    print("rejected:", e)

{'name': 'Dinakar', 'is_admin': True, 'email': None}
rejected: make_user() takes 1 positional argument but 2 were given


## Forcing positional-only — the `/` delimiter (3.8+)

The opposite: params *before* `/` cannot be passed by keyword. Used in builtins (e.g. `len(obj, /)`) and occasionally in libraries to lock in an internal parameter name.

In [6]:
def distance(x, y, /, *, scale=1.0):
    # x and y MUST be positional; scale MUST be keyword.
    return ((x ** 2 + y ** 2) ** 0.5) * scale

print(distance(3, 4))
print(distance(3, 4, scale=2))

try:
    distance(x=3, y=4)
except TypeError as e:
    print("rejected:", e)

5.0
10.0
rejected: distance() got some positional-only arguments passed as keyword arguments: 'x, y'


## Unpacking at the call site — `*` and `**`

The same syntax in a *call* means "spread this iterable / mapping into arguments."

In [7]:
def greet(name, greeting="Hello"):
    return f"{greeting}, {name}!"

args = ("Dinakar",)
kwargs = {"greeting": "Hey"}

print(greet(*args, **kwargs))      # same as greet("Dinakar", greeting="Hey")

# A common real pattern: pass through arbitrary kwargs to a wrapped function
def shouting_greet(*args, **kwargs):
    return greet(*args, **kwargs).upper()

print(shouting_greet("Dinakar"))
print(shouting_greet("Dinakar", greeting="Hi"))

Hey, Dinakar!
HELLO, DINAKAR!
HI, DINAKAR!


## Mini-exercise

1. Write `pick(d, *keys)` that returns a new dict containing only the given keys from `d`. Missing keys should be ignored. Example: `pick({"a":1,"b":2,"c":3}, "a", "c") == {"a":1, "c":3}`.
2. Write `safe_div(a, b, /, *, default=0)`. Returns `a / b`, or `default` if `b == 0`. Why does forcing `default` to be keyword-only make this safer?
3. Predict, then run:
   ```python
   def f(a, b, *, c):
       return a, b, c
   print(f(1, 2, 3))   # ?
   print(f(1, 2, c=3)) # ?
   ```

In [8]:
# your solutions here


**Takeaways**

- `*args` collects extra positionals → tuple. `**kwargs` collects extra keywords → dict.
- The same `*` / `**` syntax at a *call site* spreads.
- A bare `*` in a signature forces everything after it to be keyword-only.
- `/` (3.8+) forces everything before it to be positional-only.
- Use keyword-only for **booleans and flags** — `make_user("Dinakar", True)` is unreadable.

Next: [02_closures_lambdas.ipynb](02_closures_lambdas.ipynb)